In [1]:
# Settings: file paths and the client's block centres (Config sheet)
import glob
import pandas as pd, numpy as np
from pyproj import Transformer

GCOV_FOLDER = "../../Raw Data/GCOV Data"
SME2_FILE = "../../Raw Data/sm_avg_block_trend_20251101.0_20260926.0.xlsx"
OUTPUT_FILE = "../../Processed Data/Downscaling_Dataset_Simple.xlsx"

CONFIG_LATS = [36.82813644, 36.82618332, 36.82423401, 36.82228470]                  # block rows 0-3
CONFIG_LONS = [-120.1773834, -120.1753082, -120.1732330, -120.1711655, -120.1690903]  # block columns 0-4

In [2]:
# NASA's grid (EASE-Grid 2.0): 200.18 m cells counted from the grid's west and north edges
CELL = 36032.220840584 / 180
GRID_WEST = -17367530.445161
GRID_NORTH = 7314540.830639
to_ease = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True)     # lat/lon -> EASE-Grid metres

# Put the client's block centres on the grid
lon, lat = np.meshgrid(CONFIG_LONS, CONFIG_LATS)
x, y = to_ease.transform(lon, lat)
cols = (x - GRID_WEST) / CELL
rows = (GRID_NORTH - y) / CELL
FIRST_COL = int(cols[0, 0])
FIRST_ROW = int(rows[0, 0])

# Each centre sits in the middle of a NASA cell (decimal part 0.5), which ties SME2 block IDs to cells
distance = np.sqrt((cols % 1 - 0.5) ** 2 + (rows % 1 - 0.5) ** 2) * CELL
print("Block 0_0 = grid column", FIRST_COL, "row", FIRST_ROW)            # 28834, 14617
print(f"Centres are at most {distance.max():.2f} m from NASA cell centres")  # 0.40 m

Block 0_0 = grid column 28834 row 14617
Centres are at most 0.40 m from NASA cell centres


In [3]:
# Read the 6 radar dates. Files use UTM 10N, UTM 11N or lat/lon, so convert every file to lat/lon
utm10 = Transformer.from_crs("EPSG:32610", "EPSG:4326", always_xy=True)
utm11 = Transformer.from_crs("EPSG:32611", "EPSG:4326", always_xy=True)

all_dates = []
for hh_file in sorted(glob.glob(GCOV_FOLDER + "/GCOV_*_hhhh.xlsx")):
    hh = pd.read_excel(hh_file, index_col=0).stack()
    hv = pd.read_excel(hh_file.replace("_hhhh", "_hvhv"), index_col=0).stack()
    df = pd.DataFrame({"HH": hh, "HV": hv}).reset_index()
    df.columns = ["y", "x", "HH", "HV"]
    x = df["x"].astype(float).values
    y = df["y"].astype(float).values

    if abs(x).max() <= 180:                           # already lat/lon
        lon, lat = x, y
    else:
        lon, lat = utm10.transform(x, y)
        if abs(lon.mean() - CONFIG_LONS[0]) > 0.05:   # not on the farm, so the file is in zone 11
            lon, lat = utm11.transform(x, y)

    stamp = hh_file.split("GCOV_")[-1]                # e.g. 20260204T130902 (UTC)
    df["Longitude"] = lon
    df["Latitude"] = lat
    df["Date"] = stamp[:4] + "-" + stamp[4:6] + "-" + stamp[6:8]
    df["Pass"] = stamp[9:11] + ":" + stamp[11:13] + " UTC"
    all_dates.append(df.drop(columns=["x", "y"]))

pixels = pd.concat(all_dates, ignore_index=True)
pixels = pixels[(pixels["HH"] > 0) & (pixels["HV"] > 0)]     # dB needs values above 0
print("Pixels:", len(pixels), "| dates:", pixels["Date"].nunique())

Pixels: 43095 | dates: 6


In [4]:
# Drop each pixel into the NASA cell its centre falls in (NASA's "drop-in-bucket")
x, y = to_ease.transform(pixels["Longitude"].values, pixels["Latitude"].values)
col = (x - GRID_WEST) / CELL
row = (GRID_NORTH - y) / CELL
pixels["Col"] = np.floor(col).astype(int) - FIRST_COL
pixels["Row"] = np.floor(row).astype(int) - FIRST_ROW
pixels["X_in_block"] = col % 1          # position inside the block, 0 to 1
pixels["Y_in_block"] = row % 1

pixels = pixels[pixels["Row"].between(0, 3) & pixels["Col"].between(0, 4)].copy()
pixels["Block_ID"] = "B" + pixels["Row"].astype(str) + "_" + pixels["Col"].astype(str)

# Alignment check: pixels per block on each date (about 400 for a full block)
pixels.pivot_table(index="Date", columns="Block_ID", values="HH", aggfunc="count")

Block_ID,B0_0,B0_1,B0_2,B0_3,B0_4,B1_0,B1_1,B1_2,B1_3,B1_4,B2_0,B2_1,B2_2,B2_3,B2_4,B3_0,B3_1,B3_2,B3_3,B3_4
Date,,,,,,,,,,,,,,,,,,,,
2026-02-04,385,398,402,403,258,363,396,409,393,281,344,407,395,406,288,246,282,272,260,189
2026-02-10,385,398,402,403,258,363,396,409,393,281,344,407,395,406,288,246,282,272,260,189
2026-02-17,342,407,396,404,289,370,390,414,390,284,372,409,392,411,256,242,255,274,275,183
2026-02-28,385,398,402,403,258,363,396,409,393,281,344,407,395,406,288,246,282,272,260,189
2026-03-01,342,407,396,404,289,370,390,414,390,284,372,409,392,411,256,242,255,274,275,183
2026-03-13,357,378,399,378,273,374,396,418,396,286,374,396,418,396,286,238,252,266,252,182


In [5]:
# Fallow / almond edge: the latitude where HH brightness drops the most, found on each date
edges = {}
for date, day in pixels.groupby("Date"):
    bands = day.groupby(day["Latitude"].round(4))["HH"].agg(["mean", "size"])
    bands = bands[bands["size"] >= bands["size"].median() / 2]      # skip thin bands at the image border
    brightness = 10 * np.log10(bands["mean"])
    edges[date] = brightness.diff().idxmin()
    print(date, "edge at latitude", edges[date])

EDGE = pd.Series(edges).median()
pixels["Crop"] = np.where(pixels["Latitude"] >= EDGE, "Fallow", "Almond")
print("Edge used:", EDGE)   # 36.8258

2026-02-04 edge at latitude 36.8258
2026-02-10 edge at latitude 36.8258
2026-02-17 edge at latitude 36.8258
2026-02-28 edge at latitude 36.8258
2026-03-01 edge at latitude 36.8258
2026-03-13 edge at latitude 36.8259
Edge used: 36.8258


In [6]:
# Join NASA SME2 by UTC date and block ID
sme2 = pd.read_excel(SME2_FILE)
sme2["Date"] = pd.to_datetime(sme2["date"].astype(str), format="%Y%m%d").dt.strftime("%Y-%m-%d")
sme2 = sme2.drop(columns="date").melt(id_vars="Date", var_name="Block_ID", value_name="SME2")
sme2["Block_ID"] = "B" + sme2["Block_ID"]
sme2 = sme2[sme2["SME2"].between(0, 1)]

pixels = pixels.merge(sme2, on=["Date", "Block_ID"])
print("Pixels with SME2:", len(pixels))

Pixels with SME2: 40556


In [7]:
# Average pixels into 200 m blocks (n=1), 100 m pieces (n=2) and 50 m pieces (n=4)
# Average radar in linear power, then convert to dB
pixels["Almond"] = (pixels["Crop"] == "Almond").astype(int)
pixels["Block_HH_linear"] = pixels.groupby(["Date", "Block_ID"])["HH"].transform("mean")

tables = {}
for n in [1, 2, 4]:
    pixels["Piece_ID"] = ("P" + (pixels["Y_in_block"] * n).astype(int).astype(str)
                          + "_" + (pixels["X_in_block"] * n).astype(int).astype(str))
    t = pixels.groupby(["Date", "Pass", "Block_ID", "Piece_ID"], as_index=False).agg(
        Latitude=("Latitude", "mean"), Longitude=("Longitude", "mean"), Pixels=("HH", "count"),
        Almond_Share=("Almond", "mean"), HH_linear=("HH", "mean"), HV_linear=("HV", "mean"),
        Block_HH_linear=("Block_HH_linear", "first"), SME2=("SME2", "first"))
    t["Crop"] = np.where(t["Almond_Share"] >= 0.5, "Almond", "Fallow")
    t["HH_dB"] = 10 * np.log10(t["HH_linear"])
    t["HV_dB"] = 10 * np.log10(t["HV_linear"])
    t["HV_HH_Ratio"] = t["HV_linear"] / t["HH_linear"]
    t["Block_HH_dB"] = 10 * np.log10(t["Block_HH_linear"])
    tables[n] = t.sort_values(["Date", "Block_ID", "Piece_ID"]).reset_index(drop=True)

blocks, pieces_100, pieces_50 = tables[1], tables[2], tables[4]
print("200 m blocks:", len(blocks), "| 100 m pieces:", len(pieces_100), "| 50 m pieces:", len(pieces_50))  # 120 / 480 / 1710

200 m blocks: 120 | 100 m pieces: 480 | 50 m pieces: 1710


In [8]:
# Block type: Clean, Mixed (part almond) or Edge (under 320 pixels = less than 80% covered)
blocks["Block_Type"] = "Clean"
blocks.loc[(blocks["Almond_Share"] > 0) & (blocks["Almond_Share"] < 1), "Block_Type"] = "Mixed"
blocks.loc[blocks["Pixels"] < 320, "Block_Type"] = "Edge"

types = blocks[["Date", "Block_ID", "Block_Type"]]
pieces_100 = pieces_100.merge(types, on=["Date", "Block_ID"])
pieces_50 = pieces_50.merge(types, on=["Date", "Block_ID"])
pd.crosstab(blocks["Block_Type"], blocks["Crop"], margins=True)   # the 24 clean fallow records train the model

Crop,Almond,Fallow,All
Block_Type,,,
Clean,24,24,48
Edge,36,12,48
Mixed,0,24,24
All,60,60,120


In [9]:
# Save (separate file, so the main dataset stays as it is)
block_cols = ["Date", "Pass", "Block_ID", "Latitude", "Longitude", "Pixels", "Crop", "Block_Type",
              "HH_linear", "HV_linear", "HH_dB", "HV_dB", "HV_HH_Ratio", "SME2"]
piece_cols = ["Date", "Pass", "Block_ID", "Piece_ID", "Latitude", "Longitude", "Pixels", "Crop", "Block_Type",
              "HH_linear", "HV_linear", "HH_dB", "HV_dB", "HV_HH_Ratio", "Block_HH_dB", "SME2"]

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    blocks[block_cols].to_excel(writer, sheet_name="Blocks 200m", index=False)
    pieces_100[piece_cols].rename(columns={"SME2": "Block_SME2"}).to_excel(writer, sheet_name="Pieces 100m", index=False)
    pieces_50[piece_cols].rename(columns={"SME2": "Block_SME2"}).to_excel(writer, sheet_name="Pieces 50m", index=False)
print("Saved", OUTPUT_FILE)

Saved ../../Processed Data/Downscaling_Dataset_Simple.xlsx
